# Libraries

In [1]:
import requests
import pandas as pd
import re
from collections import Counter
import time
from datetime import datetime

In [ ]:
# ====================== MY CREDENTIALS ======================

API_KEY = "IPNE3v0fjVp3PCd5mdP0mJF3x"              
API_KEY_SECRET = "IoNoWvLiTk9kDZ3KmGxDzZ351GrtouBPtBluz6m7P5YrATwuyb"
BEARER_TOKEN = "AAAAAAAAAAAAAAAAAAAAAM228gEAAAAAmy33dsYre2uE63MJlMF4tAcwkk0%3DaGP2lEo9NquKQiAY2tDk4yYYcNMQLJqDaFPUn4cjsCN3a4dOQU"  

In [9]:
# ====================== PARAMETERS ======================
handle = "ManUtd"   

In [10]:
# Expanded fields - very useful metrics
tweet_fields = (
    "created_at,public_metrics,lang,source,"
    "entities,possibly_sensitive,conversation_id,"
    "in_reply_to_user_id,referenced_tweets"
)

user_fields = (
    "created_at,description,location,public_metrics,"
    "verified,verified_type,profile_image_url"
)

In [12]:
# Base URL for recent search (last 7 days only)
url = "https://api.X.com/2/tweets/search/recent"

headers = {
    "Authorization": f"Bearer {BEARER_TOKEN}"
}

def fetch_tweets_from_user(max_results=100):
    """
    Fetch up to 100 recent tweets from a specific user (@ManUtd).
    Handles basic pagination if needed.
    """
    all_tweets = []
    pagination_token = None
    collected = 0

    while collected < max_results:
        params = {
            "query": f"from:{handle} -is:retweet",   # Original tweets only (remove -is:retweet if you want retweets too)
            "tweet.fields": tweet_fields,
            "user.fields": user_fields,
            "expansions": "author_id",
            "max_results": min(100, max_results - collected)
        }

        if pagination_token:
            params["next_token"] = pagination_token

        response = requests.get(url, headers=headers, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            print(response.json())
            break

        data = response.json()

        if "data" not in data:
            print("No more tweets found.")
            break

        tweets = data["data"]
        includes = data.get("includes", {})
        
        # Create a user lookup dict for quick access
        users = {user["id"]: user for user in includes.get("users", [])}

        for tweet in tweets:
            author = users.get(tweet["author_id"], {})

            row = {
                "tweet_id": tweet["id"],
                "text": tweet.get("text"),
                "created_at": tweet.get("created_at"),
                "lang": tweet.get("lang"),
                "source": tweet.get("source"),
                "likes": tweet.get("public_metrics", {}).get("like_count"),
                "retweets": tweet.get("public_metrics", {}).get("retweet_count"),
                "replies": tweet.get("public_metrics", {}).get("reply_count"),
                "quotes": tweet.get("public_metrics", {}).get("quote_count"),
                "impressions": tweet.get("public_metrics", {}).get("impression_count"),
                "bookmarks": tweet.get("public_metrics", {}).get("bookmark_count"),
                "author_id": tweet["author_id"],
                "author_name": author.get("name"),
                "author_username": author.get("username"),
                "author_followers": author.get("public_metrics", {}).get("followers_count"),
                "author_following": author.get("public_metrics", {}).get("following_count"),
                "author_tweets": author.get("public_metrics", {}).get("tweet_count"),
                "author_verified": author.get("verified"),
                "is_reply": "in_reply_to_user_id" in tweet,
                "conversation_id": tweet.get("conversation_id"),
            }

            all_tweets.append(row)
            collected += 1
            
            # Handle pagination
        pagination_token = data.get("meta", {}).get("next_token")
        if not pagination_token:
            break

        time.sleep(1)  # Be respectful to rate limits

    print(f"Successfully collected {len(all_tweets)} tweets from @{handle}")
    return all_tweets

In [ ]:
# ====================== RUN THE COLLECTION ======================
if __name__ == "__main__":
    tweets_list = fetch_tweets_from_user(max_results=100)

    if tweets_list:
        # Convert to pandas DataFrame (very convenient for analysis)
        df = pd.DataFrame(tweets_list)

        # Basic cleaning / feature extraction
        df["created_at"] = pd.to_datetime(df["created_at"])
        df["date"] = df["created_at"].dt.date
        df["hour"] = df["created_at"].dt.hour
        
        # Save to CSV
        filename = f"commbank_tweets_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        df.to_csv(filename, index=False, encoding="utf-8")
        print(f"Data saved to {filename}")
        
        # Quick summary statistics
        print("\n=== Quick Summary ===")
        print(f"Total tweets: {len(df)}")
        print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")
        print(f"Average likes per tweet: {df['likes'].mean():.1f}")
        print(f"Most common hour of posting: {df['hour'].mode()[0]}:00")
        
        # Example: Top words (simple)
        all_text = " ".join(df["text"].str.lower())
        words = re.findall(r'\b\w+\b', all_text)
        word_freq = Counter(words)
        print("\nTop 10 most frequent words:")
        print(word_freq.most_common(10))

        # Display first few rows
        print("\nFirst 5 tweets:")
        print(df[["created_at", "text", "likes", "retweets"]].head())

Error: 402
{'account_id': 2039033898714136576, 'title': 'CreditsDepleted', 'detail': 'Your enrolled account [2039033898714136576] does not have any credits to fulfill this request.', 'type': 'https://api.twitter.com/2/problems/credits'}
Successfully collected 0 tweets from @ManUtd
